# 02 向量检索

把文档库向量化，用问题的向量去搜最相关的文档。

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

from src.embed import embed

/home/yixienaqi0818/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 知识库：文档 → 向量

In [2]:
docs = [
    "DeepSeek API 支持流式输出，只需在调用时设置 stream=True 参数，然后迭代接收每个 chunk。",
    "RAG（检索增强生成）通过先从外部知识库检索相关文档，再将文档与问题一起送入 LLM 来提升回答质量。",
    "Python 是一门简洁易读的编程语言，广泛用于数据分析、Web 开发和人工智能领域。",
    "北京今天天气晴朗，气温 25°C，非常适合户外活动如爬山和骑行。",
    "意大利菜以番茄、橄榄油和新鲜香草为基础，披萨和意大利面是最具代表性的美食。",
    "Transformer 架构通过自注意力机制（Self-Attention）捕捉序列中任意两个位置之间的依赖关系，是 LLM 的核心基础。",
    "使用 DeepSeek 的 Chat API 时，可以通过 system prompt 设定助手的行为风格和角色定位。",
    "索引（Index）是向量检索的核心数据结构，存储了所有文档的 embedding 向量以便快速搜索。",
]

doc_vectors = embed(docs)

print(f"文档数: {len(docs)}, 向量维度: {doc_vectors.shape[1]}")

Loading weights: 100%|██████████| 71/71 [00:00<00:00, 1079.43it/s]


文档数: 8, 向量维度: 512


## 检索：问题 → 最相关文档

In [5]:
def search(query, doc_vectors, docs, top_k=3):
    query_vec = embed([query])
    scores = cosine_similarity(query_vec, doc_vectors)[0]
    ranked = np.argsort(scores)[::-1]
    for rank, idx in enumerate(ranked[:top_k]):
        print(f"Top{rank+1} [{scores[idx]:.3f}] {docs[idx]}")
    print()

search("怎么开启流式输出？", doc_vectors, docs)
search("今天适合出门吗？", doc_vectors, docs)
search("什么是注意力机制？", doc_vectors, docs)

Top1 [0.593] DeepSeek API 支持流式输出，只需在调用时设置 stream=True 参数，然后迭代接收每个 chunk。
Top2 [0.349] 使用 DeepSeek 的 Chat API 时，可以通过 system prompt 设定助手的行为风格和角色定位。
Top3 [0.342] 索引（Index）是向量检索的核心数据结构，存储了所有文档的 embedding 向量以便快速搜索。

Top1 [0.592] 北京今天天气晴朗，气温 25°C，非常适合户外活动如爬山和骑行。
Top2 [0.316] Python 是一门简洁易读的编程语言，广泛用于数据分析、Web 开发和人工智能领域。
Top3 [0.278] 意大利菜以番茄、橄榄油和新鲜香草为基础，披萨和意大利面是最具代表性的美食。

Top1 [0.613] Transformer 架构通过自注意力机制（Self-Attention）捕捉序列中任意两个位置之间的依赖关系，是 LLM 的核心基础。
Top2 [0.437] 索引（Index）是向量检索的核心数据结构，存储了所有文档的 embedding 向量以便快速搜索。
Top3 [0.373] RAG（检索增强生成）通过先从外部知识库检索相关文档，再将文档与问题一起送入 LLM 来提升回答质量。

